# ax-js End-to-End Demo

Complete workflow: set up an Ax experiment, run Bayesian optimization, export the fitted GP model, and visualize predictions interactively — all client-side.

**Requirements**: `pip install ax-platform botorch`

## 1. Set Up Experiment

In [ ]:
from ax.api import Client
from ax.api.configs import RangeParameterConfig

client = Client()

# Branin function: 2D continuous optimization
client.configure_experiment(
    name='branin_demo',
    parameters=[
        RangeParameterConfig(name='x1', parameter_type='float', bounds=(-5.0, 10.0)),
        RangeParameterConfig(name='x2', parameter_type='float', bounds=(0.0, 15.0)),
    ],
    description='Branin benchmark for ax-js demo',
)
client.configure_optimization(objective='branin', outcome_constraints=[])
print(f'Experiment configured: {client._experiment.name}')

## 2. Run Bayesian Optimization

In [ ]:
from botorch.test_functions import Branin
import torch

branin_fn = Branin(negate=True)  # negate for maximization

# Run 25 trials (Sobol init + BO)
for i in range(25):
    params, trial_index = client.get_next_trial()
    x = torch.tensor([[params['x1'], params['x2']]])
    y = branin_fn(x).item()
    client.complete_trial(
        trial_index=trial_index,
        raw_data={'branin': y},
    )
    if (i + 1) % 5 == 0:
        print(f'  Trial {i+1}: x1={params["x1"]:.3f}, x2={params["x2"]:.3f}, y={y:.3f}')

print(f'\nCompleted {i+1} trials')

## 3. Export Model to ax-js

The export captures everything needed for client-side prediction: kernel hyperparameters, training data, input/output transforms.

In [ ]:
import sys, json
sys.path.insert(0, 'python')  # for axjs_export
from axjs_export import export_client

state = export_client(client)
print(f'Exported: {len(state["model_state"]["train_X"])} training points')
print(f'Kernel: {state["model_state"]["kernel"]["type"]}')
print(f'Outcome: {state["outcome_names"]}')

# Save to JSON (optional)
with open('demo/branin_export.json', 'w') as f:
    json.dump(state, f)
print(f'Saved to demo/branin_export.json ({len(json.dumps(state))//1024}KB)')

## 4. Visualize with ax-js

Load the ax-js JavaScript bundles and render interactive plots — all predictions computed client-side in the browser.

In [ ]:
from IPython.display import display, HTML
from pathlib import Path

# Load ax-js bundles
ax_js = Path('dist/ax.js').read_text()
viz_js = Path('dist/ax-viz.js').read_text()
state_json = json.dumps(state)

# Initialize ax-js in the notebook
display(HTML(f'<script>{ax_js}\n{viz_js}\nwindow.__AXJS_STATE__={state_json}</script>'
             '<div style="color:#888;font-size:12px">ax-js loaded.</div>'))

### 1D Slice Plots

In [ ]:
display(HTML('''
<div id="sp" style="width:100%;min-height:400px;position:relative;background:#0f0f11;border-radius:8px;overflow:visible;padding:8px"></div>
<script>(function(){
  var c=document.getElementById('sp');
  var p=new Ax.Predictor(window.__AXJS_STATE__);
  Ax.viz.renderSlicePlot(c,p,{interactive:true});
})()</script>
'''))

### 2D Response Surface

In [ ]:
display(HTML('''
<div id="rs" style="width:500px;min-height:500px;position:relative;background:#0f0f11;border-radius:8px;overflow:visible;padding:8px"></div>
<script>(function(){
  var c=document.getElementById('rs');
  var p=new Ax.Predictor(window.__AXJS_STATE__);
  Ax.viz.renderResponseSurface(c,p,{interactive:true,width:460,height:460});
})()</script>
'''))

### Leave-One-Out Cross-Validation

In [ ]:
display(HTML('''
<div id="cv" style="width:450px;min-height:450px;position:relative;background:#0f0f11;border-radius:8px;overflow:visible;padding:8px"></div>
<script>(function(){
  var c=document.getElementById('cv');
  var p=new Ax.Predictor(window.__AXJS_STATE__);
  Ax.viz.renderCrossValidation(c,p,{interactive:true,width:420,height:420});
})()</script>
'''))

### Feature Importance & Optimization Trace

In [ ]:
display(HTML('''
<div id="fi" style="width:100%;min-height:250px;position:relative;background:#0f0f11;border-radius:8px;overflow:visible;padding:8px;margin-bottom:12px"></div>
<div id="ot" style="width:700px;min-height:400px;position:relative;background:#0f0f11;border-radius:8px;overflow:visible;padding:8px"></div>
<script>(function(){
  var p=new Ax.Predictor(window.__AXJS_STATE__);
  Ax.viz.renderFeatureImportance(document.getElementById('fi'),p,{interactive:true});
  Ax.viz.renderOptimizationTrace(document.getElementById('ot'),p,{interactive:true,width:660,height:360});
})()</script>
'''))

---
## About

This notebook demonstrates the full ax-js workflow:
1. **Set up** an Ax experiment with `ax.api.Client`
2. **Run** Bayesian optimization (Sobol init → GP-based BO)
3. **Export** the fitted model to JSON via `axjs_export.export_client()`
4. **Visualize** predictions interactively — all rendering happens in JavaScript, no Python backend needed

Export this notebook to HTML with `jupyter nbconvert --to html` for a standalone page (after executing all cells).